# Retrieval Augmented Generation : Pipeline 1 is Data Ingestion & Pipeline 2 is Augmented Generation

# Langchain Document Structure
```
doc = Document(
    pageContent: str,
    metadata: {dict}
)

```

```
from langchain_core.documents import Document 

doc = Document(
    page_content = "main text content",
    metadata = {
        "source":"sample.txt",
        "pages": 100,
        "author":"Rudyard Kipling"
    }
) 

```
Documentation Page Link: https://docs.langchain.com/oss/javascript/integrations/document_loaders/index#interface

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import re
import os 


In [7]:
def load_pdfs(pdf_directory)-> list:
    """Loading all PDF's"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF's ")
    print("Loading...")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✅ Loaded {len(all_documents)}")


        except Exception as e:
            print("❌ Couldn't load documents")
            print(f"Error {e}")

    
    return all_documents



all_docs = load_pdfs("./kaggle_dataset")
print(all_docs)

Found 2 PDF's 
Loading...
Processing docum.pdf
✅ Loaded 73
Processing legaldoc.pdf
✅ Loaded 176
[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-01-29T19:33:21+05:30', 'author': 'keyush nisar', 'moddate': '2025-01-29T19:33:21+05:30', 'source': 'kaggle_dataset\\docum.pdf', 'total_pages': 73, 'page': 0, 'page_label': '1', 'source_file': 'docum.pdf', 'file_type': 'pdf'}, page_content='What is Lease Deed (for a term of years) Rent Agreement? \nA rent or lease agreement is an agreement that lays down pre-discussed terms and conditions \nunder which a property is to be rented or leased between a tenant and landlord. A lease \nagreement is essentially an agreement for leasing of an immovable property for up to 11 \nmonths. A rent agreement can be an agreement of more than a year. All important clauses \nstating the terms, conditions and promises between both the parties must be included in a \nlease or r

# Common Questions to ask:
- “What should an NDA contain?”
- “How does a lease agreement work?”
- “Generate a separation agreement”
- “What documents are needed for adoption deed?”

In [10]:
# READ THE .MD FILE TO UNDERSTAND CHUNKING
def split_documents(documents):
    """Split the documents
    in such a way that we preserve semantic meaning
    across Clauses, Sections etc. etc.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 2200,
        chunk_overlap = 300,
        length_function = len,
        separators = [
            "\n\nSECTION ",
            "\n\nARTICLE ",
            "\n\nCHAPTER ",
            "\n\n",
            ". ",
            "; ",
            " "
        ]
    )

    split_docs = []

    for doc in documents:

        text = doc.page_content

        # Before anything, create segments for major legal sections.
        sections = re.split(
            r'(?=\n(?:SECTION|Section|ARTICLE|Article|CHAPTER|Chapter))',
            text
        )


        for section in sections:
            chunks = splitter.create_documents(
                [section],
                metadatas = [doc.metadata]
            )


            split_docs.extend(chunks)

    return split_docs 


split_docs = split_documents(all_docs)
for chunk in split_docs:
    print(chunk)
    

page_content='What is Lease Deed (for a term of years) Rent Agreement? 
A rent or lease agreement is an agreement that lays down pre-discussed terms and conditions 
under which a property is to be rented or leased between a tenant and landlord. A lease 
agreement is essentially an agreement for leasing of an immovable property for up to 11 
months. A rent agreement can be an agreement of more than a year. All important clauses 
stating the terms, conditions and promises between both the parties must be included in a 
lease or rent agreement.  
Why is Lease Deed (for a term of years) Rent Agreement required? 
A lease or rent agreement is an essential document that evidences the leasing or renting of an 
immovable property. All terms relating to such lease/rent including the time period for such 
lease and the purpose for which the property must be used, among other terms, must form a 
part of a rent agreement. Such an agreement serves the purpose of a reference document for all 
the par